In [0]:
%run ./common

In [0]:
# Run ONCE after a successful FULL raw load and file verification.
# This is safe to rerun: it never rewinds an existing watermark.
import json
from pathlib import Path
from datetime import datetime, timezone
from pyspark.sql import functions as F
from delta.tables import DeltaTable

spark.conf.set('spark.sql.session.timeZone','UTC')
dbutils.widgets.text('manifest_path',
 '/Volumes/github_health/raw/bronze_data/bronze/_manifests/full_load_2026-10-08_92199a9d-c132-4525-92c9-5094e8e1f310.json')
manifest=json.loads(Path(dbutils.widgets.get('manifest_path')).read_text())
end_time=parse_iso(manifest['window_end'])
files={(x['repository'],x['entity']):x for x in manifest['files']
       if x.get('repository') in REPOS and x.get('entity') in ENTITIES}
assert len(files)==9, 'Expected 9 commit/issue/pull files in full-load manifest'
for key,item in files.items():
    f=Path(item['path'])
    assert f.is_file(),f'Missing {f}'
    assert f.stat().st_size==item['bytes'],f'Byte mismatch {f}'
    with f.open(encoding='utf-8') as inp:
        actual=sum(1 for line in inp if line.strip())
    assert actual==item['records'],f'Row mismatch {f}: {actual}'
rows=[(repo,entity,'RAW',end_time,manifest['batch_id'],now())
      for repo in REPOS for entity in ENTITIES]
source=spark.createDataFrame(rows,WM_SCHEMA)
(DeltaTable.forName(spark,WM_TABLE).alias('t')
 .merge(source.alias('s'),
        't.repository=s.repository AND t.entity=s.entity AND t.layer=s.layer')
 .whenNotMatchedInsertAll().execute())
print('Verified all nine FULL raw files; initial RAW checkpoints inserted if missing.')
display(spark.table(WM_TABLE).filter(F.col('layer')=='RAW').orderBy('repository','entity'))
